In [ ]:
import os 
import getpass
from dotenv import load_dotenv
import pydantic
from crewai import LLM, Agent, Task, Crew
from com.example.agentic.agent.LLMManager import LLMManager


llm = LLMManager.create_llm('ollama')
llm.call('Hello')

In [ ]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage

class CustomKnowledgeStorage(KnowledgeStorage):

    def __init__(self, persist_directory: str, embedder=None, collection_name=None):
        self.persist_directory = persist_directory
        self.embedder = embedder
        self.collection_name = collection_name
        super().__init__(embedder=embedder, collection_name=collection_name)
    
    def initialize_knowledge_storage(self):
        pass

In [1]:
from crewai.tools import tool
from crewai_tools import PDFSearchTool
from com.example.agentic.tools.config import _tool_config, _rag_tool_config, _embedder_config_openai
#pydantic.SkipValidation

# 1. Initialize the tool
# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
#@tool

_pdf_tool = PDFSearchTool(
    pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
    config=_rag_tool_config
)
#_pdf_tool.add(pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf')

CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf


In [2]:
results = _pdf_tool.run(query="Spark & Kafka")
results

"Relevant Content:\n\nimproved release frequency from monthly to bi-weekly, ensuring on-time delivery against stringent regulatory \n\ndeadlines. \n\n \n\n• Led Data Engineering Chapter for IB Markets Regulatory Reporting, governing 5+ regulatory \n\nframeworks (MIFID-II, RTS23, EUBR/UKBR, SSD, SFTR) — overseeing ingestion, reconciliation, and distribution \n\nof deal-store data for ~10 million+ daily trade & transaction records with full audit trail and zero regulatory \n\nbreach incidents across a 3-year delivery window. \n\n \n\n\n\n\n\n\nPage 2:\n\n \n\n• Designed and executed migration strategy from Cloudera Data Platform to Azure Databricks, leveraging Azure \n\nData Factory, ADX, and Apache Flink — reducing data processing latency by ~50%, cutting infrastructure \n\nlicensing costs by ~35%, and enabling elastic scaling for peak regulatory reporting windows without manual \n\nintervention. \n\n \n\n• Partnered with business, operations, and risk teams to identify and remediate 3 

In [ ]:
from datetime import datetime
from crewai import Crew, Agent, Task, Process
from crewai import context
from emoji import config
from com.example.agentic.tools.config import _tool_config, _embedder_config_openai

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(run_id)

from crewai_tools import FileReadTool

# Initialize tool
file_read_tool = FileReadTool(file_path='/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt')
#file_read_tool.run()

# 3. Agent and Task Definition
text_specialist = Agent(
    role='Text File Reader',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[file_read_tool],
    #llm=llm
)

# 2. Define the Agent
resume_analyst = Agent(
    role='As a resume analyst',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[],
    allow_delegation=False,
    verbose=True,
    #llm=llm
)

# 3. Define the Task
read_task = Task(
    description='Read text content',
    expected_output='text contain in the file.',
    agent=text_specialist,
    output_file=f'outputs/L004/read_task_{run_id}.txt'
)

extract_task = Task(
    description='review the content that you receive and make sure you extract skills, experience and work location',
    expected_output='A json string contain skills, experience and work location and list of domain knowledge',
    agent=text_specialist,
    context=[read_task],
    output_file=f'outputs/L004/extract_task_{run_id}.json'
)

# 4. Run the Crew
crew = Crew(
    agents=[text_specialist], 
    tasks=[read_task,extract_task],
    process=Process.sequential,
    verbose=True,
    #planning=True,
    #planning_llm="openai/llama3.2:1b-instruct-q8_0"
)
result = crew.kickoff()
print(result)

20260424_095136


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6ef99ed7-7c24-4c36-92c7-8d74456cf17c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read text content                                                                                        │
│  ID: 3d8c29fc-72bb-4ce1-82a8-3e514baa2a99                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Text File Reader                                                                                        │
│                                                                                                                 │
│  Task: Read text content                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'file_path': '/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt'}    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#21) ───────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Output: Brijesh K. Dhaker                                                                                      │
│  A2-802, Acolade Co. HSG, Kharadi, Pune 411014 • (+91) 9820937445 • brijeshdhaker@gmail.com                     │
│  https://www.linkedin.com/in/brijesh-k-dhaker-08458872                                                          │
│  Professional Summary                                                                                           │
│  Senior technology executive with 21+ years of leadership across Investment Banking and Assets & Wealth         │
│  Management at Tier-1 global banks (UBS, JPMorgan, Goldman Sachs). Proven track record of building and          │
│  scaling Agentic AI, GenAI, and RAG-based data platforms — currently leading UBS's Client Risk Rating           │
│  modernization using LangChain, LangGraph, MCP, and A2A to deliver a high-volume, low-latency compliance        │
│  intelligence platform. Deep expertise in Data Engineering, Cloud Migration (Azure Databricks, AWS), and IB     │
│  Regulatory Reporting (MIFID-II, SFTR, EUBR/UKBR), with a strong record of delivering zero-breach regulatory    │
│  programs and Financial Crime Prevention platforms at enterprise scale. Experienced in managing                 │
│  cross-functional                                                                                               │
│  teams of 10–12 engineers, driving AI strategy, data governance, and digital transformation from architecture   │
│  through production.                                                                                            │
│  Core Competencies                                                                                              │
│  AI/ML Strategy & Roadmap • Agentic AI Platform Engineering • Data Architecture & Governance • Financial Crime  │
│  Compliance • IB Regulatory Reporting • Cloud Migration & Modernization • Team Building & Leadership •          │
│  Stakeholder Management • Digital Transformation                                                                │
│  Experience                                                                                                     │
│  DIRECTOR – AI & DATA ENGINEERING | UBS BUSINESS SOLUTIONS | PUNE | APR 2025 – PRESENT                          │
│  •AI & Innovation Lead, Financial Crime Prevention Compliance Reporting: Architected and leading UBS's          │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#22) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'file_path': '/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt'}    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#22) ───────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Output: Brijesh K. Dhaker                                                                                      │
│  A2-802, Acolade Co. HSG, Kharadi, Pune 411014 • (+91) 9820937445 • brijeshdhaker@gmail.com                     │
│  https://www.linkedin.com/in/brijesh-k-dhaker-08458872                                                          │
│  Professional Summary                                                                                           │
│  Senior technology executive with 21+ years of leadership across Investment Banking and Assets & Wealth         │
│  Management at Tier-1 global banks (UBS, JPMorgan, Goldman Sachs). Proven track record of building and          │
│  scaling Agentic AI, GenAI, and RAG-based data platforms — currently leading UBS's Client Risk Rating           │
│  modernization using LangChain, LangGraph, MCP, and A2A to deliver a high-volume, low-latency compliance        │
│  intelligence platform. Deep expertise in Data Engineering, Cloud Migration (Azure Databricks, AWS), and IB     │
│  Regulatory Reporting (MIFID-II, SFTR, EUBR/UKBR), with a strong record of delivering zero-breach regulatory    │
│  programs and Financial Crime Prevention platforms at enterprise scale. Experienced in managing                 │
│  cross-functional                                                                                               │
│  teams of 10–12 engineers, driving AI strategy, data governance, and digital transformation from architecture   │
│  through production.                                                                                            │
│  Core Competencies                                                                                              │
│  AI/ML Strategy & Roadmap • Agentic AI Platform Engineering • Data Architecture & Governance • Financial Crime  │
│  Compliance • IB Regulatory Reporting • Cloud Migration & Modernization • Team Building & Leadership •          │
│  Stakeholder Management • Digital Transformation                                                                │
│  Experience                                                                                                     │
│  DIRECTOR – AI & DATA ENGINEERING | UBS BUSINESS SOLUTIONS | PUNE | APR 2025 – PRESENT                          │
│  •AI & Innovation Lead, Financial Crime Prevention Compliance Reporting: Architected and leading UBS's          │
│  first Agentic AI & RAG-based modernization of the Client Risk Rating (CRR) platform — transforming a legacy    │
│  brown-field processing stack into a high-volume, low-latency, green-field platform using LangChain,            │
│  LangGraphs, and LLM-powered agents integrated via A2A and MCP protocols, targeting a 60%+ reduction in         │
│  manual compliance review time across Financial Crime Prevention workflows.                                     │
│  •Leading end-to-end Solution Architecture for Client Risk Rating (CRR) and SVC applications                    │
│  across microservices, distributed systems, and agentic AI orchestration layers — defining technical design     │
│  standards, enforcing code quality processes, and integrating Vector DB and Embeddings to enable semantic       │
│  search and context-aware risk scoring at enterprise scale.                                                     │
│  DIRECTOR – DATA STRATEGY & MODERNIZATION | UBS BUSINESS SOLUTIONS | PUNE | APR 2022 –                          │
│  MAR 2025                                              

╭──────────────────────────────────────── 🔧 Tool Execution Started (#23) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'file_path': '/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt'}    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#23) ───────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Output: Brijesh K. Dhaker                                                                                      │
│  A2-802, Acolade Co. HSG, Kharadi, Pune 411014 • (+91) 9820937445 • brijeshdhaker@gmail.com                     │
│  https://www.linkedin.com/in/brijesh-k-dhaker-08458872                                                          │
│  Professional Summary                                                                                           │
│  Senior technology executive with 21+ years of leadership across Investment Banking and Assets & Wealth         │
│  Management at Tier-1 global banks (UBS, JPMorgan, Goldman Sachs). Proven track record of building and          │
│  scaling Agentic AI, GenAI, and RAG-based data platforms — currently leading UBS's Client Risk Rating           │
│  modernization using LangChain, LangGraph, MCP, and A2A to deliver a high-volume, low-latency compliance        │
│  intelligence platform. Deep expertise in Data Engineering, Cloud Migration (Azure Databricks, AWS), and IB     │
│  Regulatory Reporting (MIFID-II, SFTR, EUBR/UKBR), with a strong record of delivering zero-breach regulatory    │
│  programs and Financial Crime Prevention platforms at enterprise scale. Experienced in managing                 │
│  cross-functional                                                                                               │
│  teams of 10–12 engineers, driving AI strategy, data governance, and digital transformation from architecture   │
│  through production.                                                                                            │
│  Core Competencies                                                                                              │
│  AI/ML Strategy & Roadmap • Agentic AI Platform Engineering • Data Architecture & Governance • Financial Crime  │
│  Compliance • IB Regulatory Reporting • Cloud Migration & Modernization • Team Building & Leadership •          │
│  Stakeholder Management • Digital Transformation                                                                │
│  Experience                                                                                                     │
│  DIRECTOR – AI & DATA ENGINEERING | UBS BUSINESS SOLUTIONS | PUNE | APR 2025 – PRESENT                          │
│  •AI & Innovation Lead, Financial Crime Prevention Compliance Reporting: Architected and leading UBS's          │
│  first Agentic AI & RAG-based modernization of the Client Risk Rating (CRR) platform — transforming a legacy    │
│  brown-field processing stack into a high-volume, low-latency, green-field platform using LangChain,            │
│  LangGraphs, and LLM-powered agents integrated via A2A and MCP protocols, targeting a 60%+ reduction in         │
│  manual compliance review time across Financial Crime Prevention workflows.                                     │
│  •Leading end-to-end Solution Architecture for Client Risk Rating (CRR) and SVC applications                    │
│  across microservices, distributed systems, and agentic AI orchestration layers — defining technical design     │
│  standards, enforcing code quality processes, and integrating Vector DB and Embeddings to enable semantic       │
│  search and context-aware risk scoring at enterprise scale.                                                     │
│  DIRECTOR – DATA STRATEGY & MODERNIZATION | UBS BUSINESS SOLUTIONS | PUNE | APR 2022 –                          │
│  MAR 2025                                              